In [ ]:
## 基础配置

In [1]:
import json
import logging
import os
from datetime import datetime, timezone
from logging import INFO

import asyncio
from dotenv import load_dotenv

from graphiti_core import Graphiti
from graphiti_core.nodes import EpisodeType
from graphiti_core.search.search_config_recipes import NODE_HYBRID_SEARCH_RRF
from graphiti_core import Graphiti
from graphiti_core.llm_client.openai_generic_client import OpenAIGenericClient
from graphiti_core.llm_client.config import LLMConfig
from graphiti_core.embedder.openai import OpenAIEmbedder, OpenAIEmbedderConfig
from graphiti_core.cross_encoder.openai_reranker_client import OpenAIRerankerClient

from dotenv import load_dotenv

load_dotenv()

logging.basicConfig(
    level=INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s',
    datefmt='%Y-%m-%d %H:%M:%S',
)
logger = logging.getLogger(__name__)

In [2]:
llm_config = LLMConfig(
    api_key=os.getenv("DASHSCOPE_API_KEY"),
    model="qwen-plus",  # e.g., "mistral-large-latest"
    small_model="qwen-flash",  # e.g., "mistral-small-latest"
    base_url=os.getenv("DASH_SCOPE_BASE_URL"),  # e.g., "https://api.mistral.ai/v1"
)

graphiti = Graphiti(
    "bolt://localhost:7687",
    "neo4j",
    "password",
    llm_client=OpenAIGenericClient(config=llm_config),
    embedder=OpenAIEmbedder(
        config=OpenAIEmbedderConfig(
            api_key=os.getenv("DASHSCOPE_API_KEY"),
            embedding_model="text-embedding-v4",  # e.g., "mistral-embed"
            base_url=os.getenv("DASH_SCOPE_BASE_URL"),
        )
    ),
    # cross_encoder=OpenAIRerankerClient(
    #     config=LLMConfig(
    #         api_key=os.getenv("DASHSCOPE_API_KEY"),
    #         model="text-embedding-bge-reranker-v2-m3",  # Use smaller model for reranking
    #         base_url="http://127.0.0.1:1234/v1"
    #     )
    # ),
    cross_encoder=OpenAIRerankerClient(
        config=LLMConfig(
            api_key=os.getenv("DASHSCOPE_API_KEY"),
            model="qwen-flash",  # 阿里云官方兼容重排模型
            base_url=os.getenv("DASH_SCOPE_BASE_URL"),  # 正确兼容地址
        )
    )
)

2026-04-20 10:33:35 - neo4j.notifications - INFO - Received notification from DBMS server: <GqlStatusObject gql_status='00NA0', status_description="note: successful completion - index or constraint already exists. The command 'CREATE RANGE INDEX episode_group_id IF NOT EXISTS FOR (e:Episodic) ON (e.group_id)' has no effect. The index or constraint specified by 'RANGE INDEX episode_group_id FOR (e:Episodic) ON (e.group_id)' already exists.", position=None, raw_classification='SCHEMA', classification=<NotificationClassification.SCHEMA: 'SCHEMA'>, raw_severity='INFORMATION', severity=<NotificationSeverity.INFORMATION: 'INFORMATION'>, diagnostic_record={'_classification': 'SCHEMA', '_severity': 'INFORMATION', 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: 'CREATE INDEX episode_group_id IF NOT EXISTS FOR (n:Episodic) ON (n.group_id)'
2026-04-20 10:33:35 - neo4j.notifications - INFO - Received notification from DBMS server: <GqlStatusObject gql_status='00NA0', sta

### 自定义节点和关系类型

In [3]:
from pydantic import BaseModel, Field
from datetime import datetime
from typing import Optional


# Custom Entity Types
class Person(BaseModel):
    """A person entity with biographical information."""
    age: Optional[int] = Field(None, description="Age of the person")
    occupation: Optional[str] = Field(None, description="Current occupation")
    location: Optional[str] = Field(None, description="Current location")
    birth_date: Optional[datetime] = Field(None, description="Date of birth")


class Company(BaseModel):
    """A business organization."""
    industry: Optional[str] = Field(None, description="Primary industry")
    founded_year: Optional[int] = Field(None, description="Year company was founded")
    headquarters: Optional[str] = Field(None, description="Location of headquarters")
    employee_count: Optional[int] = Field(None, description="Number of employees")


class Product(BaseModel):
    """A product or service."""
    category: Optional[str] = Field(None, description="Product category")
    price: Optional[float] = Field(None, description="Price in USD")
    release_date: Optional[datetime] = Field(None, description="Product release date")


# Custom Edge Types
class Employment(BaseModel):
    """Employment relationship between a person and company."""
    position: Optional[str] = Field(None, description="Job title or position")
    start_date: Optional[datetime] = Field(None, description="Employment start date")
    end_date: Optional[datetime] = Field(None, description="Employment end date")
    salary: Optional[float] = Field(None, description="Annual salary in USD")
    is_current: Optional[bool] = Field(None, description="Whether employment is current")


class Investment(BaseModel):
    """Investment relationship between entities."""
    amount: Optional[float] = Field(None, description="Investment amount in USD")
    investment_type: Optional[str] = Field(None, description="Type of investment (equity, debt, etc.)")
    stake_percentage: Optional[float] = Field(None, description="Percentage ownership")
    investment_date: Optional[datetime] = Field(None, description="Date of investment")


class Partnership(BaseModel):
    """Partnership relationship between companies."""
    partnership_type: Optional[str] = Field(None, description="Type of partnership")
    duration: Optional[str] = Field(None, description="Expected duration")
    deal_value: Optional[float] = Field(None, description="Financial value of partnership")


entity_types = {
    "Person": Person,
    "Company": Company,
    "Product": Product
}

edge_types = {
    "Employment": Employment,
    "Investment": Investment,
    "Partnership": Partnership
}

edge_type_map = {
    ("Person", "Company"): ["Employment"],
    ("Company", "Company"): ["Partnership", "Investment"],
    ("Person", "Person"): ["Partnership"],
    ("Entity", "Entity"): ["Investment"],  # Apply to any entity type
}

In [7]:
# await graphiti.add_episode(
#     name="Business Update",
#     episode_body="Sarah joined TechCorp as CTO in January 2023 with a $200K salary. TechCorp partnered with DataCorp in a $5M deal.",
#     source_description="Business news",
#     reference_time=datetime.now(),
#     entity_types=entity_types,
#     edge_types=edge_types,
#     edge_type_map=edge_type_map
# )
# 2. 新增CEO雇佣
await graphiti.add_episode(
    name="Executive Hire",
    episode_body="Mike was appointed CEO of DataCorp in March 2022 with a $250,000 annual salary.",
    source_description="Corporate announcement",
    reference_time=datetime.now(),
    entity_types=entity_types,
    edge_types=edge_types,
    edge_type_map=edge_type_map
)

# 3. 投资事件
await graphiti.add_episode(
    name="Investment Round",
    episode_body="VentureFund invested $15M in TechCorp in December 2023, acquiring 12% equity stake.",
    source_description="Financial news",
    reference_time=datetime.now(),
    entity_types=entity_types,
    edge_types=edge_types,
    edge_type_map=edge_type_map
)

# 4. 产品发布
await graphiti.add_episode(
    name="Product Launch",
    episode_body="TechCorp released its AI Analytics Suite product on June 2024, priced at $499 per license.",
    source_description="Product announcement",
    reference_time=datetime.now(),
    entity_types=entity_types,
    edge_types=edge_types,
    edge_type_map=edge_type_map
)

# 5. 三方战略合作
await graphiti.add_episode(
    name="Strategic Alliance",
    episode_body="DataCorp, TechCorp and GreenEnergy formed a sustainability partnership worth $8M for 3 years.",
    source_description="Press release",
    reference_time=datetime.now(),
    entity_types=entity_types,
    edge_types=edge_types,
    edge_type_map=edge_type_map
)

# 6. 员工离职
await graphiti.add_episode(
    name="Employee Resignation",
    episode_body="Emma left her position as Data Engineer at DataCorp in February 2024 with a $130K salary.",
    source_description="Personnel update",
    reference_time=datetime.now(),
    entity_types=entity_types,
    edge_types=edge_types,
    edge_type_map=edge_type_map
)

# 7. 大额融资
await graphiti.add_episode(
    name="Series A Funding",
    episode_body="GreenEnergy secured $32M in Series A investment from GlobalCapital in September 2023.",
    source_description="Funding news",
    reference_time=datetime.now(),
    entity_types=entity_types,
    edge_types=edge_types,
    edge_type_map=edge_type_map
)

# 8. 跨公司合作 + 人事
await graphiti.add_episode(
    name="Business Expansion",
    episode_body="James became CFO of GreenEnergy in 2024. GreenEnergy also signed a $6M partnership with TechCorp.",
    source_description="Business news",
    reference_time=datetime.now(),
    entity_types=entity_types,
    edge_types=edge_types,
    edge_type_map=edge_type_map
)


2026-04-20 10:43:03 - httpx - INFO - HTTP Request: POST https://dashscope.aliyuncs.com/compatible-mode/v1/chat/completions "HTTP/1.1 200 OK"
2026-04-20 10:43:04 - httpx - INFO - HTTP Request: POST https://dashscope.aliyuncs.com/compatible-mode/v1/embeddings "HTTP/1.1 200 OK"
2026-04-20 10:43:04 - httpx - INFO - HTTP Request: POST https://dashscope.aliyuncs.com/compatible-mode/v1/embeddings "HTTP/1.1 200 OK"
2026-04-20 10:43:05 - httpx - INFO - HTTP Request: POST https://dashscope.aliyuncs.com/compatible-mode/v1/chat/completions "HTTP/1.1 200 OK"
2026-04-20 10:43:07 - httpx - INFO - HTTP Request: POST https://dashscope.aliyuncs.com/compatible-mode/v1/chat/completions "HTTP/1.1 200 OK"
2026-04-20 10:43:08 - httpx - INFO - HTTP Request: POST https://dashscope.aliyuncs.com/compatible-mode/v1/embeddings "HTTP/1.1 200 OK"
2026-04-20 10:43:08 - httpx - INFO - HTTP Request: POST https://dashscope.aliyuncs.com/compatible-mode/v1/embeddings "HTTP/1.1 200 OK"
2026-04-20 10:43:08 - httpx - INFO - 

AddEpisodeResults(episode=EpisodicNode(uuid='60a4f31f-ef38-44b2-983f-5126746baee0', name='Business Expansion', group_id='', labels=[], created_at=datetime.datetime(2026, 4, 20, 2, 44, 5, 388348, tzinfo=datetime.timezone.utc), source=<EpisodeType.message: 'message'>, source_description='Business news', content='James became CFO of GreenEnergy in 2024. GreenEnergy also signed a $6M partnership with TechCorp.', valid_at=datetime.datetime(2026, 4, 20, 10, 44, 5, 388348), entity_edges=['c74a8087-939c-47fc-abb4-b2608be55cd9', '5e33d1c6-7387-4c6a-9e67-20486c470984']), episodic_edges=[EpisodicEdge(uuid='836e6dea-2d34-4809-8598-82441d6ce029', group_id='', source_node_uuid='60a4f31f-ef38-44b2-983f-5126746baee0', target_node_uuid='1f8e68cd-5e88-4cd5-9160-9efb5f84e3fb', created_at=datetime.datetime(2026, 4, 20, 2, 44, 5, 388348, tzinfo=datetime.timezone.utc)), EpisodicEdge(uuid='35c7e3ab-82db-4d59-b76a-28dec9a55960', group_id='', source_node_uuid='60a4f31f-ef38-44b2-983f-5126746baee0', target_node

In [8]:
from graphiti_core.search.search_filters import SearchFilters

# Search for only specific entity types
search_filter = SearchFilters(
    node_labels=["Person", "Company"]  # Only return Person and Company entities
)

results = await graphiti.search_(
    query="Who works at tech companies?",
    search_filter=search_filter
)

# Search for only specific edge types
search_filter = SearchFilters(
    edge_types=["Employment", "Partnership"]  # Only return Employment and Partnership edges
)

results = await graphiti.search_(
    query="Tell me about business relationships",
    search_filter=search_filter
)


2026-04-20 10:45:38 - httpx - INFO - HTTP Request: POST https://dashscope.aliyuncs.com/compatible-mode/v1/embeddings "HTTP/1.1 200 OK"
2026-04-20 10:45:39 - httpx - INFO - HTTP Request: POST https://dashscope.aliyuncs.com/compatible-mode/v1/chat/completions "HTTP/1.1 200 OK"
2026-04-20 10:45:39 - httpx - INFO - HTTP Request: POST https://dashscope.aliyuncs.com/compatible-mode/v1/chat/completions "HTTP/1.1 200 OK"
2026-04-20 10:45:39 - httpx - INFO - HTTP Request: POST https://dashscope.aliyuncs.com/compatible-mode/v1/chat/completions "HTTP/1.1 200 OK"
2026-04-20 10:45:39 - httpx - INFO - HTTP Request: POST https://dashscope.aliyuncs.com/compatible-mode/v1/chat/completions "HTTP/1.1 200 OK"
2026-04-20 10:45:39 - httpx - INFO - HTTP Request: POST https://dashscope.aliyuncs.com/compatible-mode/v1/chat/completions "HTTP/1.1 200 OK"
2026-04-20 10:45:39 - httpx - INFO - HTTP Request: POST https://dashscope.aliyuncs.com/compatible-mode/v1/chat/completions "HTTP/1.1 200 OK"
2026-04-20 10:45:39

In [9]:
from pprint import pprint
pprint(results)

SearchResults(edges=[], edge_reranker_scores=[], nodes=[EntityNode(uuid='ae1d301c-c1b7-41d6-bc7c-966c203f3570', name='DataCorp', group_id='', labels=['Entity', 'Company'], created_at=datetime.datetime(2026, 4, 20, 2, 33, 37, 491035, tzinfo=<UTC>), name_embedding=None, summary='TechCorp partnered with DataCorp in a $5M deal\nMike was appointed CEO of DataCorp in March 2022 with a $250,000 annual salary.\nDataCorp, TechCorp and GreenEnergy formed a sustainability partnership worth $8M for 3 years\nDataCorp, TechCorp and GreenEnergy formed a sustainability partnership worth $8M for 3 years\nEmma left her position as Data Engineer at DataCorp in February 2024 with a $130K salary', attributes={}), EntityNode(uuid='0dbeee4e-feba-40dc-aa11-c0be8c3d5ec6', name='VentureFund', group_id='', labels=['Entity', 'Company'], created_at=datetime.datetime(2026, 4, 20, 2, 43, 13, 500977, tzinfo=<UTC>), name_embedding=None, summary='VentureFund invested $15M in TechCorp in December 2023, acquiring a 12% e

In [10]:
await graphiti.build_communities()

2026-04-20 10:47:51 - httpx - INFO - HTTP Request: POST https://dashscope.aliyuncs.com/compatible-mode/v1/chat/completions "HTTP/1.1 200 OK"
2026-04-20 10:47:51 - httpx - INFO - HTTP Request: POST https://dashscope.aliyuncs.com/compatible-mode/v1/chat/completions "HTTP/1.1 200 OK"
2026-04-20 10:47:51 - httpx - INFO - HTTP Request: POST https://dashscope.aliyuncs.com/compatible-mode/v1/chat/completions "HTTP/1.1 200 OK"
2026-04-20 10:47:52 - httpx - INFO - HTTP Request: POST https://dashscope.aliyuncs.com/compatible-mode/v1/chat/completions "HTTP/1.1 200 OK"
2026-04-20 10:47:52 - httpx - INFO - HTTP Request: POST https://dashscope.aliyuncs.com/compatible-mode/v1/chat/completions "HTTP/1.1 200 OK"
2026-04-20 10:47:52 - httpx - INFO - HTTP Request: POST https://dashscope.aliyuncs.com/compatible-mode/v1/chat/completions "HTTP/1.1 200 OK"
2026-04-20 10:47:53 - httpx - INFO - HTTP Request: POST https://dashscope.aliyuncs.com/compatible-mode/v1/chat/completions "HTTP/1.1 200 OK"
2026-04-20 10

([CommunityNode(uuid='5911b9ae-fe19-4d41-9cf6-15eae996e96c', name='The summary details recent financial investments, partnerships, and product launches involving TechCorp, DataCorp, and GreenEnergy, including funding amounts, equity stakes, alliance values, and pricing for a new AI Analytics Suite.', group_id='', labels=['Community'], created_at=datetime.datetime(2026, 4, 20, 2, 47, 59, 788326, tzinfo=datetime.timezone.utc), name_embedding=[-0.007104264572262764, 0.03412462770938873, -0.03234289959073067, 0.051881514489650726, 0.025940757244825363, 0.06788687407970428, -0.004454320762306452, 0.06915521621704102, 0.0028141492512077093, 0.08516057580709457, -0.001679807435721159, -0.0003791363269556314, 0.0273148026317358, 0.03986239805817604, -0.09385782480239868, 0.045147184282541275, 0.06867203861474991, -0.010690370574593544, -0.018119271844625473, -0.01769648864865303, 0.08159711956977844, 0.01784748211503029, 0.08081194758415222, -0.033581048250198364, 0.0023819291964173317, 0.0089